# 04. Error analysis and forest diagnostics

Use the four original frozen forests and saved features. Inspect JECAM first, the three retained supplements separately, and the revised combined pool last. The external predictions remain probability based. Hard tree votes, impurity importance, and the internal OOB diagnostics are additional evidence about the forests.

The analysis uses local archives and performs no fitting or imagery retrieval. Use trusted frozen models. To regenerate the report, select a new output directory.


In [ ]:
from pathlib import Path
import sys

root = next(path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (path / "pyproject.toml").is_file())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

from crop_pipeline.training.error_analysis import inspect_batch3

training_run = root / "pipeline_outputs/training/run_fa40f5d6bb31"
testing_run = root / "pipeline_outputs/testing/run_81a0d0a2812e"
preprocessing = root / "pipeline_outputs/preprocessing/external_xxu0iz4o"
parcel_audit = root / "pipeline_outputs/test_audit_2026-11-19/parcel_audit.json"
output = root / "pipeline_outputs/testing/run_81a0d0a2812e_batch3"


## Run and review

The OOB calculation reconstructs the weighted half-size bootstrap from the saved tree seeds and verifies the unique in-bag training count at every tree root. It reports pixel OOB, parcel summaries from pixel OOB, and a separate whole-parcel OOB diagnostic. The latter uses trees that omitted every pixel of a parcel. These are internal diagnostics.

The external tables trace errors to parcel IDs and show imagery support, reference dates, boundary proximity, and votes. Missing mask counts and unresolved error causes are recorded as unavailable or unknown. The cabbage trace uses the saved cycle-selection audit.


In [ ]:
from IPython.display import Markdown, display

report = output / "report.md"
if not (output / "inputs.json").is_file():
    report = inspect_batch3(training_run, testing_run, preprocessing, parcel_audit, output)
display(Markdown(report.read_text(encoding="utf-8")))
print("Batch 3 tables:", output)


## Inspect individual parcels

The campaign/crop table reports the confusion counts for every reference crop. Change the campaign below to inspect another test population or set it to `combined` to inspect all retained parcels.


In [ ]:
from crop_pipeline.notebook_display import show_table
from crop_pipeline.windows.seasons import read_table

campaign = "jecam"
cohort = "fixed_matched"
summary = read_table(output / "campaign_crop_errors.csv")
show_table([row for row in summary if row["population"] == campaign and row["model"] == cohort],
           title="Campaign and crop errors")

examples = read_table(output / "representative_parcels.csv")
columns = ["partition_group", "population", "reference_crop", "prediction", "usable_pixels",
           "reference_date_in_window", "min_pixel_center_to_original_boundary_m",
           "ndvi_peak_bin", "hard_vote_winner", "max_hard_vote_fraction", "error_cause"]
show_table([{key: row[key] for key in columns} for row in examples
            if row["model"] == cohort and (campaign == "combined" or row["population"] == campaign)],
           title="Traceable correct and incorrect examples")
